# How many neighbors? Guided k-NN classification with Palmer Penguins

In this activity, you will build a model that predicts a penguin's species from four body measurements. The main goal is not to obtain the largest possible score. It is to practice an honest modeling workflow and use evidence to explain when a model is underfitting, overfitting, or likely to generalize.

By the end of this notebook, you should be able to:

1. reserve an untouched test set and use cross-validation only within the training data;
2. explain why distance-based models need features on comparable scales;
3. connect the number of neighbors to model flexibility, overfitting, underfitting, bias, and variance;
4. tune `n_neighbors`, `weights`, and distance geometry without using the test set; and
5. interpret validation results, a confusion matrix, and the neighbors behind one prediction.

### What you will submit

Run the completed notebook from top to bottom. Submit the code, plots, and written responses in the marked response areas. Use your own results in your explanations; do not report a score without interpreting it.

## The validation plan

We will give each part of the data one job.

| Data | Job | May it influence a modeling choice? |
|---|---|---|
| Training folds | Fit the imputer, scaler, and k-NN classifier | Yes |
| Validation folds | Compare settings and select hyperparameters | Yes |
| Held-out test set | Estimate performance on new data once | No |

A five-fold cross-validation splitter will repeatedly create training and validation folds from `X_train`. Five folds are a practical compromise: fewer folds fit on less data and can give a more pessimistic estimate, while many highly overlapping folds cost more and can make the estimate more variable. We will keep the folds fixed for every candidate. The held-out test set will not enter cross-validation, plots used for selection, or hyperparameter tuning. If we try many settings on the test set and keep the best one, the test score becomes optimistically biased and is no longer an honest estimate of performance on new data.

> **Two different k's:** `n_neighbors` is the number of neighbors used by k-NN. `n_splits` is the number of folds used by cross-validation. They solve different problems even though both are often called *k*.

## 0. Setup

The setup is supplied. `sns.load_dataset` downloads the public Palmer Penguins data the first time it is used.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 20)

## 1. Inspect the data before modeling

The Palmer Penguins data contain observations of Adélie, Chinstrap, and Gentoo penguins from islands in the Palmer Archipelago, Antarctica. Our prediction question is:

> **Can body measurements predict a penguin's species for a new penguin from the same setting?**

### Task 1 — Audit the data

Run the next cells. Look for the number of observations, the class distribution, missing values, feature scales, and visible separation or overlap among species. Do not remove missing rows yet; preprocessing will be part of the model.

In [ ]:
penguins = sns.load_dataset("penguins")

print("Shape:", penguins.shape)
display(penguins.head())
display(penguins.dtypes.to_frame("dtype"))

In [ ]:
audit = pd.DataFrame({
    "missing_count": penguins.isna().sum(),
    "missing_fraction": penguins.isna().mean(),
    "n_unique": penguins.nunique(dropna=True),
})
display(audit)

print("Species counts:")
display(penguins["species"].value_counts().to_frame("count"))

In [ ]:
measurement_columns = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

sns.pairplot(
    penguins,
    vars=measurement_columns,
    hue="species",
    corner=True,
    plot_kws={"alpha": 0.65, "s": 35},
)
plt.show()

**Response 1.** Record three observations: one about class balance, one about missingness, and one about separation or overlap in the pair plot. Which species appears hardest to separate using only these measurements, and what evidence supports your answer?

**Response:** _Write your answer here._

## 2. Define the modeling frame and reserve the test set

We will use the four continuous measurements as predictors and `species` as the target. This keeps the meaning of distance visible. We are intentionally excluding `island`, `sex`, and `year`; including them would define a different prediction problem and would require decisions about how categorical indicators contribute to distance.

### Task 2 — Separate predictors from the target

Create `X` from the four names in `measurement_columns`. Create `y` from the target column. Do not call `dropna()` and do not standardize the full dataset.

In [ ]:
# TODO: select the four measurement columns and copy them into X.
# TODO: select the species column and copy it into y.

assert list(X.columns) == measurement_columns
assert y.name == "species"
print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

### Task 3 — Make one stratified train/test split

Use 20% of the observations for the held-out test set. Set `random_state=RANDOM_STATE` and stratify by the target. Stratification helps each species appear in roughly the same proportion in both parts.

After this cell, all model and hyperparameter choices must use only `X_train` and `y_train` until the final-test section explicitly tells you otherwise.

In [ ]:
# TODO: create X_train, X_test, y_train, and y_test with train_test_split.
# Use test_size=0.20, random_state=RANDOM_STATE, and stratify=y.

print("Training rows:", X_train.shape[0])
print("Held-out test rows:", X_test.shape[0])

In [ ]:
split_check = pd.DataFrame({
    "all": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).sort_index()
display(split_check.round(3))

assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.shape[0] + X_test.shape[0] == X.shape[0]

**Response 2.** Why do we split before learning medians or scaling parameters? Explain what information would leak if an imputer and scaler were fit on all observations. Also explain why the test set must not be used to choose the number of neighbors.

**Response:** _Write your answer here._

## 3. Establish a validation design and a baseline

A model should improve on a simple reference. `DummyClassifier(strategy="most_frequent")` always predicts the most common training-fold species. Because the species counts are not equal, report both:

- **accuracy:** the fraction of all predictions that are correct; and
- **balanced accuracy:** the mean recall across species, giving each species equal weight.

We will use shuffled, stratified five-fold cross-validation. Each observation in `X_train` is used for validation once. The held-out test set remains untouched.

### Task 4 — Cross-validate the baseline

Complete `cross_validate` with the baseline, training data, `cv`, `scoring`, and `return_train_score=True`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}

baseline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])

# TODO: cross-validate baseline using only X_train and y_train.
baseline_scores = cross_validate(
    # TODO
)

baseline_summary = pd.Series({
    "validation_accuracy_mean": baseline_scores["test_accuracy"].mean(),
    "validation_accuracy_sd": baseline_scores["test_accuracy"].std(),
    "validation_balanced_accuracy_mean": baseline_scores["test_balanced_accuracy"].mean(),
    "validation_balanced_accuracy_sd": baseline_scores["test_balanced_accuracy"].std(),
})
display(baseline_summary.round(3))

**Response 3.** Why is the baseline's balanced accuracy less flattering than its ordinary accuracy? What would you conclude if a k-NN model's validation score were only slightly better than this baseline?

**Response:** _Write your answer here._

## 4. Distance depends on preprocessing

k-NN classifies a new observation from nearby training observations. Under Euclidean distance, a difference of 500 grams contributes far more to a raw distance than a difference of 10 millimeters. That numerical dominance comes from units, not necessarily from greater predictive importance.

`StandardScaler` subtracts a training-fold mean and divides by a training-fold standard deviation. `SimpleImputer` fills missing measurements using medians learned from the training fold. Putting both inside a `Pipeline` makes cross-validation refit them separately in every fold and prevents validation information from leaking into training.

### Task 5 — Examine the raw scales

Run the summary and compare the standard deviations and ranges.

In [ ]:
scale_audit = X_train.agg(["min", "max", "std"]).T
scale_audit["range"] = scale_audit["max"] - scale_audit["min"]
display(scale_audit.round(2))

**Response 4.** Without scaling, which measurement is likely to dominate Euclidean distance? Use the output to justify your answer. Why does a larger numeric range not automatically mean that a variable should matter more?

**Response:** _Write your answer here._

### Task 6 — Build unscaled and scaled k-NN pipelines

Use median imputation in both pipelines. Add standardization only to `scaled_knn`. Use a five-neighbor classifier with Minkowski distance and `p=2` (Euclidean distance) in both. Keeping everything else fixed isolates the effect of scaling.

In [ ]:
unscaled_knn = Pipeline(steps=[
    ("imputer", None),  # TODO: median imputation
    ("knn", None),      # TODO: 5-neighbor classifier, metric="minkowski", p=2
])

scaled_knn = Pipeline(steps=[
    ("imputer", None),  # TODO: median imputation
    ("scaler", None),   # TODO: standardization
    ("knn", None),      # TODO: same classifier as above
])

### Task 7 — Compare scaling with the same folds

Cross-validate both pipelines with `X_train`, `y_train`, `cv`, and `scoring`. Request training scores so you can examine both fit and generalization. Using the same folds makes the comparison less sensitive to fold-to-fold differences.

In [ ]:
scaling_rows = []
for name, model in {"unscaled": unscaled_knn, "scaled": scaled_knn}.items():
    # TODO: cross-validate model using the requested data, folds, metrics, and training scores.
    scores = cross_validate(
        # TODO
    )
    scaling_rows.append({
        "pipeline": name,
        "training_accuracy": scores["train_accuracy"].mean(),
        "validation_accuracy": scores["test_accuracy"].mean(),
        "validation_accuracy_sd": scores["test_accuracy"].std(),
        "validation_balanced_accuracy": scores["test_balanced_accuracy"].mean(),
    })

scaling_comparison = pd.DataFrame(scaling_rows).set_index("pipeline")
display(scaling_comparison.round(3))

**Response 5.** How did scaling change mean validation accuracy and balanced accuracy? Is the difference large compared with fold-to-fold variability? Explain why scaling can change the neighbors even though it does not change the order of values within any one feature.

**Response:** _Write your answer here._

## 5. Model flexibility, overfitting, and underfitting

The number of neighbors controls the flexibility of k-NN.

| `n_neighbors` | Decision boundary | Typical training behavior | Main risk | Bias–variance tendency |
|---|---|---|---|---|
| Very small | Jagged and highly local | Very low training error | Overfitting | Lower bias, higher variance |
| Moderate | Uses a broader local pattern | Training and validation errors can both be low | Better balance | Intermediate |
| Very large | Smooth; may approach the overall class proportions | Training and validation errors can both be high | Underfitting | Higher bias, lower variance |

**Bias** is systematic error from a model whose assumptions are too restrictive. **Variance** is how much a fitted model would change across different training samples from the same population. Overfitting is associated with high variance: a model follows peculiarities of its training sample and does much better on training data than validation data. Underfitting is associated with high bias: the model is too inflexible, so both training and validation performance are poor.

A training–validation gap is a useful symptom, but it is not a direct mathematical measurement of bias or variance. Some prediction error is also irreducible: no value of `n_neighbors` can remove noise, measurement error, or overlap between species.

### Task 8 — Trace the fit across neighbor counts

> **Classroom checkpoint:** Before running the code, sketch the training- and validation-error curves you expect as the neighbor count increases. Compare your sketch with a partner and explain your reasoning.

Then evaluate the listed values using only the training data. For each value, clone `scaled_knn`, set `knn__n_neighbors`, and cross-validate it.

In [ ]:
candidate_k = [1, 3, 5, 7, 9, 13, 17, 25, 35, 51, 75, 101]
k_rows = []

for k in candidate_k:
    # TODO: clone scaled_knn and set knn__n_neighbors=k.
    
    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
    )
    k_rows.append({
        "n_neighbors": k,
        "training_accuracy_mean": scores["train_accuracy"].mean(),
        "validation_accuracy_mean": scores["test_accuracy"].mean(),
        "validation_accuracy_sd": scores["test_accuracy"].std(),
        "validation_balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
    })

k_results = pd.DataFrame(k_rows)
display(k_results.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    k_results["n_neighbors"],
    1 - k_results["training_accuracy_mean"],
    marker="o",
    label="mean training error",
)
ax.errorbar(
    k_results["n_neighbors"],
    1 - k_results["validation_accuracy_mean"],
    yerr=k_results["validation_accuracy_sd"],
    marker="o",
    capsize=3,
    label="mean validation error ± 1 SD",
)
ax.set(
    xlabel="Number of neighbors",
    ylabel="Classification error (1 - accuracy)",
    title="Model flexibility: training versus validation error",
)
ax.set_xticks(candidate_k)
ax.legend()
ax.grid(alpha=0.25)
plt.show()

**Response 6.** Use specific neighbor counts and values from the table or plot.

1. Where do you see the clearest evidence of overfitting? Describe the training–validation gap.
2. Where do you see evidence that the model is beginning to underfit? Describe what happens to both errors.
3. Which region gives the best validation performance? If several settings are within about one standard deviation, why might a somewhat larger `n_neighbors` be a reasonable tie-breaker?
4. Connect your observations explicitly to bias and variance.

**Response:** _Write your answer here._

## 6. Tune k-NN hyperparameters inside the training data

The neighbor count is not the only modeling choice. We will tune three hyperparameters:

- `n_neighbors`: smaller values are generally more flexible; larger values are smoother.
- `weights`: `"uniform"` gives each neighbor one vote, while `"distance"` gives closer observations more influence. Distance weighting can make the fit more local and may increase variance.
- `p`: with Minkowski distance, `p=1` is Manhattan distance and `p=2` is Euclidean distance. This changes the geometry of a neighborhood; neither value is universally best or simply 'more complex.'

`GridSearchCV` will fit every combination using the same five-fold design. We will select by balanced accuracy so that performance on the smaller species class is not hidden by the larger classes. Preprocessing remains inside the pipeline, so every candidate learns imputation and scaling only from its training fold.

### Task 9 — Run the grid search

Build `param_grid` with all `candidate_k` values, both weighting rules, and `p` values 1 and 2. The double-underscore names refer to parameters inside the pipeline's `knn` step. Then fit the search using only `X_train` and `y_train`.

In [ ]:
param_grid = {
    # TODO: "knn__n_neighbors"
    # TODO: "knn__weights"
    # TODO: "knn__p"
}

required_parameters = {"knn__n_neighbors", "knn__weights", "knn__p"}
assert set(param_grid) == required_parameters

search = GridSearchCV(
    estimator=scaled_knn,
    param_grid=param_grid,
    scoring=scoring,
    refit="balanced_accuracy",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
)

# TODO: fit search using only the training data.

In [ ]:
tuning_results = pd.DataFrame(search.cv_results_)
tuning_results["balanced_accuracy_gap"] = (
    tuning_results["mean_train_balanced_accuracy"]
    - tuning_results["mean_test_balanced_accuracy"]
)

result_columns = [
    "param_knn__n_neighbors",
    "param_knn__weights",
    "param_knn__p",
    "mean_train_balanced_accuracy",
    "mean_test_balanced_accuracy",
    "std_test_balanced_accuracy",
    "balanced_accuracy_gap",
    "mean_test_accuracy",
]

top_results = (
    tuning_results
    .sort_values("rank_test_balanced_accuracy")
    .loc[:, result_columns]
    .head(10)
)
display(top_results.round(3))
print("Selected settings:", search.best_params_)
print("Mean CV balanced accuracy:", round(search.best_score_, 3))

**Response 7.** Interpret the search rather than only copying `best_params_`.

1. How does the selected model compare with nearby alternatives, considering both the mean and standard deviation?
2. Does its training–validation gap suggest serious overfitting?
3. How did `weights` and `p` affect results? Avoid claiming that a tiny difference is meaningful when fold-to-fold variability is larger.
4. Which quantities in this workflow are hyperparameters chosen by you or the search, and which quantities are learned from training data?

**Response:** _Write your answer here._

## 7. Evaluate once on the held-out test set

`GridSearchCV` has already refit the selected pipeline on all of `X_train` and `y_train`; it has still never seen `X_test`. Before running the next cell, write down the approximate test performance you expect from cross-validation.

### Task 10 — Make the final predictions

Assign the refitted best estimator to `best_knn`, predict `X_test` once, and report accuracy, balanced accuracy, a classification report, and a confusion matrix. After viewing these results, do not change the model and present the same test score as an untouched estimate.

In [ ]:
best_knn = search.best_estimator_

# TODO: use best_knn to create final_predictions for X_test.

print("Cross-validation balanced accuracy:", round(search.best_score_, 3))
print("Test accuracy:", round(accuracy_score(y_test, final_predictions), 3))
print("Test balanced accuracy:", round(balanced_accuracy_score(y_test, final_predictions), 3))
print()
print(classification_report(y_test, final_predictions))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_predictions,
    display_labels=best_knn.classes_,
    cmap="Blues",
)
plt.title("Selected k-NN pipeline: held-out test set")
plt.show()

**Response 8.** Compare the held-out balanced accuracy with the cross-validation estimate and its variability. Use the confusion matrix or classification report to identify which species is hardest to classify and describe a specific error pattern. If every test case is correct, explain why this small test set still does not establish perfect future performance. Does the result suggest obvious overfitting, underfitting, or neither? State the evidence and acknowledge uncertainty from the small test set.

Suppose you now change a hyperparameter because of this test result. Why could you no longer describe this same score as a final, untouched test estimate?

**Response:** _Write your answer here._

## 8. Interpret one prediction through its neighbors

k-NN does not learn a coefficient for each feature. A useful local interpretation is to inspect the training observations nearest to a test case after the fitted imputer and scaler have transformed it.

The cell below selects the first misclassified test observation, or the first test observation if every prediction was correct. It then displays up to ten of the neighbors used by the selected classifier. The reported distances are in standardized feature space, not in millimeters or grams.

In [ ]:
misclassified_positions = np.flatnonzero(final_predictions != y_test.to_numpy())
case_position = int(misclassified_positions[0]) if len(misclassified_positions) else 0

query = X_test.iloc[[case_position]]
query_imputed = best_knn.named_steps["imputer"].transform(query)
query_scaled = best_knn.named_steps["scaler"].transform(query_imputed)
distances, neighbor_positions = best_knn.named_steps["knn"].kneighbors(query_scaled)

neighbors = X_train.iloc[neighbor_positions[0]].copy()
neighbors.insert(0, "species", y_train.iloc[neighbor_positions[0]].to_numpy())
neighbors["standardized_distance"] = distances[0]

print("Actual species:", y_test.iloc[case_position])
print("Predicted species:", final_predictions[case_position])
print("Selected weighting rule:", best_knn.named_steps["knn"].weights)
display(query)
display(neighbors.head(10))
print("Species counts among all selected neighbors:")
display(neighbors["species"].value_counts().to_frame("neighbor_count"))

**Response 9.** Explain this prediction using the nearby training observations. Does the neighborhood make the case look easy, ambiguous, or surprising? If distance weighting was selected, explain why raw neighbor counts do not completely determine the vote. Finally, explain why these neighbors support a local predictive interpretation but do not prove that changing a body measurement would cause the species to change.

**Response:** _Write your answer here._

## 9. Modeling checklist

Before finishing, verify that your notebook follows these practices:

- The split was stratified, and the test set was reserved before preprocessing.
- Missing-value imputation and scaling stayed inside every model pipeline.
- The baseline and all candidate models used the same cross-validation folds.
- Hyperparameters were chosen from validation performance on the training data, not test performance.
- Training and validation scores were compared to diagnose fit.
- Fold variability and class-specific errors were interpreted, not hidden behind one mean accuracy.
- The held-out test set was evaluated once, after selection.
- Predictive neighbors were not interpreted as causal effects.

## 10. Final synthesis

Write 6–9 sentences that tell the modeling story rather than listing isolated facts. Include:

1. why the pipeline imputed and standardized within each training fold;
2. how the selected model compared with the baseline;
3. how changing `n_neighbors` exposed overfitting, underfitting, and the bias–variance tradeoff;
4. why validation—not the test set—was used to choose hyperparameters; and
5. what the final test results do and do not justify about future penguins.

**Final response:** _Write your synthesis here._

### Optional extension

Define one new question before changing the model. For example, compare a scientifically motivated subset of measurements or add `sex` and `island` using a `ColumnTransformer` and one-hot encoding. Keep the test set out of the comparison: make the change, evaluate it with the existing cross-validation design on the training data, and explain how the new feature representation changes the meaning of distance. A fresh test set or nested cross-validation would be needed for a new unbiased performance estimate after further exploration.